In [1]:
"""
E-Commerce Sales & Customer Analytics Dashboard
Olist Brazilian E-Commerce Dataset

Pipeline: loads raw CSVs into SQLite, runs the 5 core business-question
queries, exports results for Tableau, then runs a cohort retention
analysis in Pandas and exports that too.

Run this top to bottom in Jupyter (each ### SECTION ### can be its own cell),
or as a single script: python code_clean.py
"""

import os
import pandas as pd
import sqlite3

# ============================================================
# SECTION 1: Load raw CSVs into a SQLite database
# ============================================================

DATA_DIR = 'archive'      # folder containing the raw Kaggle CSVs
DB_PATH = 'olist.db'
RESULTS_DIR = 'results'

os.makedirs(RESULTS_DIR, exist_ok=True)

conn = sqlite3.connect(DB_PATH)

raw_files = {
    'orders': f'{DATA_DIR}/olist_orders_dataset.csv',
    'order_items': f'{DATA_DIR}/olist_order_items_dataset.csv',
    'customers': f'{DATA_DIR}/olist_customers_dataset.csv',
    'products': f'{DATA_DIR}/olist_products_dataset.csv',
    'reviews': f'{DATA_DIR}/olist_order_reviews_dataset.csv',
    'payments': f'{DATA_DIR}/olist_order_payments_dataset.csv',
}

for table_name, path in raw_files.items():
    df = pd.read_csv(path)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Loaded {table_name}: {len(df)} rows")


# ============================================================
# SECTION 2: Run the 5 core SQL queries and export each as CSV
# ============================================================

queries = {

    # Q1: Monthly revenue trend
    'q1_revenue': """
        SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS month,
               SUM(oi.price) AS total_revenue,
               COUNT(DISTINCT o.order_id) AS num_orders
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY month
        ORDER BY month
    """,

    # Q2: Repeat purchase rate (joined via customer_unique_id, since
    # Olist assigns a new customer_id per order)
    'q2_repeat': """
        SELECT order_count, COUNT(*) AS num_customers
        FROM (
            SELECT c.customer_unique_id,
                   COUNT(DISTINCT o.order_id) AS order_count
            FROM orders o
            JOIN customers c ON o.customer_id = c.customer_id
            GROUP BY c.customer_unique_id
        )
        GROUP BY order_count
        ORDER BY order_count
    """,

    # Q3: Top 15 product categories by revenue
    'q3_categories': """
        SELECT p.product_category_name,
               SUM(oi.price) AS total_revenue,
               COUNT(*) AS num_items_sold
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        GROUP BY p.product_category_name
        ORDER BY total_revenue DESC
        LIMIT 15
    """,

    # Q4: Revenue by customer state
    'q4_states': """
        SELECT c.customer_state, SUM(oi.price) AS total_revenue
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY c.customer_state
        ORDER BY total_revenue DESC
    """,

    # Q5: Delivery delay vs. review score
    'q5_delivery_reviews': """
        SELECT r.review_score,
               AVG(julianday(o.order_delivered_customer_date)
                   - julianday(o.order_estimated_delivery_date)) AS avg_delay_days,
               COUNT(*) AS num_orders
        FROM orders o
        JOIN reviews r ON o.order_id = r.order_id
        WHERE o.order_delivered_customer_date IS NOT NULL
        GROUP BY r.review_score
        ORDER BY r.review_score
    """,
}

for name, sql in queries.items():
    df = pd.read_sql(sql, conn)
    df.to_csv(f'{RESULTS_DIR}/{name}.csv', index=False)
    print(f"Saved {name}: {len(df)} rows")


# ============================================================
# SECTION 3: Cohort retention analysis (Pandas)
# ============================================================

orders = pd.read_sql("""
    SELECT o.order_id, c.customer_unique_id, o.order_purchase_timestamp
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
""", conn)

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')

# each customer's first purchase month = their cohort
first_purchase = (
    orders.groupby('customer_unique_id')['order_month']
    .min()
    .reset_index()
    .rename(columns={'order_month': 'cohort_month'})
)

orders = orders.merge(first_purchase, on='customer_unique_id')
orders['months_since_first'] = (
    orders['order_month'] - orders['cohort_month']
).apply(lambda x: x.n)

# pivot: rows = cohort month, columns = months since first purchase,
# values = number of unique customers still active
cohort_counts = (
    orders.groupby(['cohort_month', 'months_since_first'])['customer_unique_id']
    .nunique()
    .reset_index()
)
cohort_pivot = cohort_counts.pivot(
    index='cohort_month', columns='months_since_first', values='customer_unique_id'
)
cohort_pivot.to_csv(f'{RESULTS_DIR}/cohort_retention.csv')

# convert to % retained, relative to each cohort's original size (month 0)
cohort_size = cohort_pivot[0]
retention_pct = cohort_pivot.divide(cohort_size, axis=0) * 100
retention_pct.to_csv(f'{RESULTS_DIR}/cohort_retention_pct.csv')

print("Saved cohort_retention.csv and cohort_retention_pct.csv")
print(retention_pct.round(1))


# ============================================================
# SECTION 4: Reshape retention % into long format for Tableau
# ============================================================

retention_long = (
    retention_pct.reset_index()
    .melt(id_vars='cohort_month', var_name='months_since_first', value_name='retention_pct')
    .dropna(subset=['retention_pct'])
)
retention_long['months_since_first'] = retention_long['months_since_first'].astype(int)
retention_long.to_csv(f'{RESULTS_DIR}/retention_heatmap_long.csv', index=False)

print(f"Saved retention_heatmap_long.csv: {len(retention_long)} rows")

conn.close()
print("\nAll done. Six CSVs are in the 'results/' folder, ready to connect in Tableau:")
print("  q1_revenue.csv, q2_repeat.csv, q3_categories.csv,")
print("  q4_states.csv, q5_delivery_reviews.csv, retention_heatmap_long.csv")

Loaded orders: 99441 rows
Loaded order_items: 112650 rows
Loaded customers: 99441 rows
Loaded products: 32951 rows
Loaded reviews: 99224 rows
Loaded payments: 103886 rows
Saved q1_revenue: 23 rows
Saved q2_repeat: 9 rows
Saved q3_categories: 15 rows
Saved q4_states: 27 rows
Saved q5_delivery_reviews: 5 rows
Saved cohort_retention.csv and cohort_retention_pct.csv
months_since_first     0      1    2    3    4    5    6    7    8    9    10  \
cohort_month                                                                    
2016-09             100.0    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2016-10             100.0    NaN  NaN  NaN  NaN  NaN  0.4  NaN  NaN  0.4  NaN   
2016-12             100.0  100.0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2017-01             100.0    0.3  0.3  0.1  0.4  0.1  0.4  0.1  0.1  NaN  0.4   
2017-02             100.0    0.2  0.3  0.1  0.4  0.1  0.2  0.2  0.1  0.2  0.1   
2017-03             100.0    0.4  0.4  0.4  0.4  0.2  0.2  0.3  0.3 